In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
REPO_ROOT = Path.cwd().resolve().parent.parent
os.chdir(str(REPO_ROOT))

In [ ]:
os.getcwd()

In [ ]:
sys.path

In [ ]:
import optuna
import pandas as pd
import yaml
from train import train
from tuning import tune

from wip_pred.data_loader.fetch_data import fetch_raw_data
from wip_pred.inference.predict import predict
from wip_pred.prep_data.normalize_data import normalize_df, transform_df
from wip_pred.prep_data.prepare_data import get_df_of_completed_lots

In [ ]:
# Settings
with open("config/pred_template/core_template.yaml", "r") as f:
    core_config = yaml.safe_load(f)

with open("config/pred_template/train_template.yaml", "r") as f:
    train_config = yaml.safe_load(f)

train_config["training"]["save_dir"] = None
train_config["training"]["log_dir"] = None

pred_start_date = "2025-01-01"
pred_end_date = "2025-04-30"
val_days = 7
search_time_range = [270]
search_update_freq = [30]
search_tune_interval = [30]

In [ ]:
# prepare data
original_df = df = fetch_raw_data(embed_config=core_config["embed"])
original_df = normalize_df(
    original_df,
    "lot_id",
    "parent_lot_id",
    "product_id",
    "stage",
    "lot_status",
    "current_wafer_qty",
    "created_at",
    truncate_stages=["QC1-PACK", "Inventory", "PA2-PHOTO", "PAD-PHOTO"],
    do_time_binning=True,
    normalize_by_date=core_config["embed"]["use_given_emb"],
    print_details=False,
)
df_of_completed_lots = get_df_of_completed_lots(original_df)
val_days = pd.Timedelta(days=val_days)

In [ ]:
def search_stage(trial):
    return {
        "training.batch_size": trial.suggest_categorical(
            "training.batch_size", [32, 64, 128, 256, 512, 1024]
        ),
        "optimizer.params.lr": trial.suggest_float("optimizer.params.lr", 1e-5, 1e-2, log=True),
        "optimizer.params.weight_decay": trial.suggest_float(
            "optimizer.params.weight_decay", 1e-6, 1e-2, log=True
        ),
    }  # Important: key and param name must be the same!


pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=15, interval_steps=1)

In [ ]:
# check
for uf in search_update_freq:
    for ti in search_tune_interval:
        if ti % uf != 0:
            raise ValueError(f"tune_interval({ti}) 必须是 update_freq({uf}) 的整数倍")

In [ ]:
results = pd.DataFrame()

for time_range in search_time_range:
    for update_freq in search_update_freq:
        for tune_interval in search_tune_interval:
            if tune_interval % update_freq != 0:
                continue

            print(
                f"time_range: {time_range} | update_freq: {update_freq} | tune_interval: {tune_interval}"
            )
            time_to_update = pd.Timestamp(pred_start_date)
            time_to_tune = pd.Timestamp(pred_start_date)

            for pred_date in pd.date_range(pred_start_date, pred_end_date):
                start_time = pred_date - val_days - pd.Timedelta(days=time_range)
                df = original_df[
                    (original_df["created_at"] > start_time)
                    & (original_df["created_at"] <= pred_date)
                ].copy()
                # pred_ts = pd.Timestamp(pred_date) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
                # df = original_df[(original_df['created_at'] > start_time) & (original_df['created_at'] <= pred_ts)].copy()

                if pred_date == time_to_tune:
                    print(f"Time to tune hyper-params: {pred_date}")
                    val_end = pred_date
                    train_end = pred_date - val_days
                    tune(
                        core_config,
                        train_config,
                        df,
                        train_end,
                        val_end,
                        search_stage,
                        pruner,
                        n_jobs_stage1=1,
                        n_jobs_stage2=3,
                    )
                    time_to_tune += pd.Timedelta(days=tune_interval)

                if pred_date == time_to_update:
                    print(f"Time to update the model: {pred_date}")
                    val_end = pred_date
                    train_end = pred_date - val_days
                    best_model, preprocessor = train(
                        core_config, train_config, df=df, train_end=train_end, val_end=val_end
                    )
                    time_to_update = time_to_update + pd.Timedelta(days=update_freq)
                # predict
                lot_ids = df_of_completed_lots[
                    (df_of_completed_lots["end_time"] > pred_date)
                    & (df_of_completed_lots["start_time"] <= pred_date)
                ].index.tolist()  # all lots in process need to be predicted
                df = transform_df(df, core_config["embed"], pred_date, preprocessor=preprocessor)  # type: ignore
                # df = transform_df(df, core_config['embed'], pred_ts, preprocessor=preprocessor)

                pred_vals = predict(best_model, df, core_config, lot_ids)
                new_df = pd.DataFrame({"lot_id": pred_vals.index, "pred": pred_vals})
                new_df["time_range"] = time_range
                new_df["update_freq"] = update_freq
                new_df["tune_interval"] = tune_interval
                new_df["pred_date"] = pred_date
                new_df["train_end"] = train_end
                results = pd.concat([results, new_df], axis=0, ignore_index=True)

In [ ]:
results["pred_end_time"] = results["pred_date"] + pd.to_timedelta(results["pred"], unit="D")
results

In [ ]:
def prepare_for_analysis(df_results, df_of_completed_lots):
    df_results = pd.merge(
        left=df_results,
        right=df_of_completed_lots[["start_time", "end_time", "LT", "product_id"]],
        left_on="lot_id",
        right_index=True,
        how="left",
    )

    # absolute error
    df_results["error"] = (df_results["pred_end_time"] - df_results["end_time"]).abs().dt.days

    # absolute progress
    df_results["absolute_progress"] = (df_results["pred_date"] - df_results["start_time"]).dt.days

    # relative progress
    df_results["relative_progress"] = df_results["absolute_progress"] / df_results["LT"] * 100

    return df_results

In [ ]:
df_results = prepare_for_analysis(results, df_of_completed_lots)

In [ ]:
df_results.to_csv("outputs/metrics/covergence_experiments_enriched.csv", index=False)

In [ ]:
df_results.head(3)

In [ ]:
# Statistics of industrial metrics
for time_range in search_time_range:
    for update_freq in search_update_freq:
        for tune_interval in search_tune_interval:
            if tune_interval % update_freq != 0:
                continue
            print(
                f"time_range: {time_range} | update_freq: {update_freq} | tune_interval: {tune_interval}"
            )
            df_results_filtered = df_results[
                (df_results["time_range"] == time_range)
                & (df_results["update_freq"] == update_freq)
                & (df_results["tune_interval"] == tune_interval)
            ]
            # error at 1/3 LT
            tmp = df_results_filtered[
                df_results_filtered["pred_date"]
                == (
                    df_results_filtered["start_time"]
                    + 1 / 3 * df_results_filtered["LT"] * pd.Timedelta(days=1)
                ).dt.normalize()
            ]
            error_1 = (tmp["pred_end_time"] - tmp["end_time"]).abs().dt.days.mean()
            # error at all time
            error_2 = (
                (df_results_filtered["pred_end_time"] - df_results_filtered["end_time"])
                .abs()
                .dt.days.mean()
            )
            print(f"    mean error at 1/3LT: {error_1:.3f}   mean error of all time: {error_2:.3f}")
            # fig = analyze_results(df_results, time_range, update_freq, tune_interval,
            #                     interpolate=False)
            # fig.show()
        print()

In [ ]:
len(tmp)

In [ ]:
diff = (tmp["pred_end_time"] - tmp["end_time"]).abs().dt.days

In [ ]:
diff.quantile([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])